# 🦴 Bone Fracture Detection — EfficientNet + Grad-CAM
**Projet Machine Learning — Master**

Dataset : [Fracture Multi-Region X-ray Data (Kaggle)](https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data)

Architecture : EfficientNet-B3 (Transfer Learning)

---

## Cellule 1 — Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q kaggle timm grad-cam torchmetrics scikit-learn matplotlib seaborn pillow

## Cellule 2 — Configuration Kaggle API & téléchargement du dataset

In [ ]:
import os

# ⚠️ Remplace par tes vraies credentials Kaggle
os.environ['KAGGLE_USERNAME'] = 'TON_USERNAME'
os.environ['KAGGLE_KEY'] = 'TA_CLE_API'

# Téléchargement du dataset
!kaggle datasets download -d bmadushanirodrigo/fracture-multi-region-x-ray-data -p /content/data --unzip
print('✅ Dataset téléchargé')

## Cellule 3 — Vérification de la structure du dataset

In [ ]:
import os

DATA_ROOT = '/content/data/Fracture Multi-Region X-ray Data'

for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.exists(split_path):
        print(f'❌ Dossier manquant : {split_path}')
        continue
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        count = len(os.listdir(cls_path))
        print(f'{split}/{cls}: {count} images')

## Cellule 4 — Imports et configuration GPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device utilisé : {DEVICE}')

# Hyperparamètres
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 15
LR         = 1e-4
NUM_CLASSES = 2

## Cellule 5 — EDA : distribution des classes & exemples d'images

In [ ]:
# Comptage des images par split et par classe
splits = ['train', 'val', 'test']
stats  = {}

for split in splits:
    stats[split] = {}
    split_path = os.path.join(DATA_ROOT, split)
    for cls in os.listdir(split_path):
        stats[split][cls] = len(os.listdir(os.path.join(split_path, cls)))

# Graphique distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, split in enumerate(splits):
    classes = list(stats[split].keys())
    counts  = list(stats[split].values())
    axes[i].bar(classes, counts, color=['#e74c3c', '#2ecc71'])
    axes[i].set_title(f'{split.upper()} split')
    axes[i].set_ylabel('Nombre d\'images')
    for j, v in enumerate(counts):
        axes[i].text(j, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Distribution des classes par split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA sauvegardé')

## Cellule 6 — Exemples d'images du dataset

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
split_path = os.path.join(DATA_ROOT, 'train')

for row, cls in enumerate(os.listdir(split_path)):
    cls_path = os.path.join(split_path, cls)
    imgs = os.listdir(cls_path)[:4]
    for col, img_name in enumerate(imgs):
        img = Image.open(os.path.join(cls_path, img_name)).convert('RGB')
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].set_title(cls, fontweight='bold')
        axes[row][col].axis('off')

plt.suptitle('Exemples de radiographies — Train set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/eda_exemples.png', dpi=150, bbox_inches='tight')
plt.show()

## Cellule 7 — Prétraitement & DataLoaders

In [ ]:
# Transformations — ImageNet normalization pour EfficientNet
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_ROOT, 'val'),   transform=val_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_ROOT, 'test'),  transform=val_transforms)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Classes
CLASS_NAMES = train_dataset.classes
print(f'✅ Classes : {CLASS_NAMES}')
print(f'   Train : {len(train_dataset)} | Val : {len(val_dataset)} | Test : {len(test_dataset)}')

## Cellule 8 — Construction du modèle EfficientNet-B3

In [ ]:
def build_model(num_classes=2):
    """
    EfficientNet-B3 pré-entraîné sur ImageNet.
    On gèle les couches de base et on remplace le classifier.
    """
    model = timm.create_model('efficientnet_b3', pretrained=True)

    # Geler tous les paramètres
    for param in model.parameters():
        param.requires_grad = False

    # Remplacer le classifier final
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )

    return model

model = build_model(NUM_CLASSES).to(DEVICE)
print('✅ Modèle EfficientNet-B3 chargé')
print(f'   Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Cellule 9 — Gestion du déséquilibre de classes (class weights)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Calcul des poids de classe pour gérer le déséquilibre
labels_array = np.array(train_dataset.targets)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_array),
    y=labels_array
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'✅ Poids de classes : {dict(zip(CLASS_NAMES, class_weights.round(3)))}')

# Perte + optimiseur
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## Cellule 10 — Boucle d'entraînement

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} — Loss: {loss.item():.4f}')

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels, all_probs


# Historique
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
SAVE_PATH = '/content/best_model.pth'

print('🚀 Début de l\'entraînement...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'=== Époque {epoch}/{NUM_EPOCHS} ===')

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss,   val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    # Sauvegarde du meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES
        }, SAVE_PATH)
        print(f'  ✅ Meilleur modèle sauvegardé (val_acc={val_acc:.4f})')
    print()

print(f'🎯 Entraînement terminé. Meilleure val acc : {best_val_acc:.4f}')

## Cellule 11 — Fine-tuning : dégeler les dernières couches

In [ ]:
# Charger le meilleur modèle
checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])

# Dégeler les dernières couches pour le fine-tuning
for name, param in model.named_parameters():
    if 'blocks.6' in name or 'blocks.7' in name or 'classifier' in name or 'conv_head' in name:
        param.requires_grad = True

print(f'✅ Paramètres entraînables après fine-tuning : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Optimizer avec LR plus petit pour le fine-tuning
optimizer_ft = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=5)

print('🚀 Fine-tuning sur 5 époques...')
for epoch in range(1, 6):
    print(f'=== Fine-tuning Époque {epoch}/5 ===')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_ft, DEVICE)
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler_ft.step()

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': val_acc, 'class_names': CLASS_NAMES}, SAVE_PATH)
        print(f'  ✅ Modèle mis à jour (val_acc={val_acc:.4f})')

print(f'🎯 Fine-tuning terminé. Meilleure val acc : {best_val_acc:.4f}')

## Cellule 12 — Courbes d'apprentissage

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss')
ax1.set_title('Loss par époque', fontweight='bold')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc')
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc')
ax2.set_title('Accuracy par époque', fontweight='bold')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Courbes d\'apprentissage — EfficientNet-B3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cellule 13 — Évaluation finale sur le test set

In [ ]:
# Charger le meilleur modèle
checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Évaluation sur le test set
_, test_acc, test_preds, test_labels, test_probs = evaluate(model, test_loader, criterion, DEVICE)

print('=' * 50)
print('📊 RÉSULTATS SUR LE TEST SET')
print('=' * 50)
print(f'Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print()
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

# AUC-ROC
auc = roc_auc_score(test_labels, test_probs)
print(f'AUC-ROC : {auc:.4f}')

## Cellule 14 — Matrice de confusion

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.title('Matrice de Confusion — Test Set', fontsize=14, fontweight='bold')
plt.ylabel('Vraie classe')
plt.xlabel('Classe prédite')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Cellule 15 — Grad-CAM : Visualisation des zones d'attention

In [ ]:
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Identifier la dernière couche conv pour Grad-CAM
target_layers = [model.conv_head]

cam = GradCAM(model=model, target_layers=target_layers)

def generate_gradcam(model, cam, image_path, class_names, device):
    """
    Génère une heatmap Grad-CAM pour une image donnée.
    Retourne : image originale, heatmap, image superposée, classe prédite, confidence
    """
    # Prétraitement
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = transform(img_pil).unsqueeze(0).to(device)

    # Prédiction
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probs  = torch.softmax(output, dim=1)
        pred_class = probs.argmax().item()
        confidence = probs[0][pred_class].item()

    # Grad-CAM
    targets   = [ClassifierOutputTarget(pred_class)]
    grayscale = cam(input_tensor=img_tensor, targets=targets)[0]

    # Image normalisée pour superposition
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    cam_image = show_cam_on_image(img_np.astype(np.float32), grayscale, use_rgb=True)

    return img_pil, grayscale, cam_image, class_names[pred_class], confidence

print('✅ Grad-CAM prêt')

## Cellule 16 — Visualisation Grad-CAM sur des exemples

In [ ]:
# Sélection d'exemples pour la visualisation
sample_images = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATA_ROOT, 'test', cls)
    imgs = os.listdir(cls_path)[:3]
    for img_name in imgs:
        sample_images.append((os.path.join(cls_path, img_name), cls))

fig, axes = plt.subplots(len(sample_images), 3, figsize=(15, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = [axes]

for i, (img_path, true_cls) in enumerate(sample_images[:6]):
    try:
        orig, heatmap, cam_img, pred_cls, conf = generate_gradcam(model, cam, img_path, CLASS_NAMES, DEVICE)

        axes[i][0].imshow(orig)
        axes[i][0].set_title(f'Original\nVraie classe: {true_cls}', fontweight='bold')
        axes[i][0].axis('off')

        axes[i][1].imshow(heatmap, cmap='jet')
        axes[i][1].set_title('Heatmap Grad-CAM', fontweight='bold')
        axes[i][1].axis('off')

        color = 'green' if pred_cls == true_cls else 'red'
        axes[i][2].imshow(cam_img)
        axes[i][2].set_title(f'Superposition\nPrédit: {pred_cls} ({conf:.2%})', fontweight='bold', color=color)
        axes[i][2].axis('off')
    except Exception as e:
        print(f'Erreur sur {img_path}: {e}')

plt.suptitle('Visualisations Grad-CAM', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/gradcam_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM sauvegardé')

## Cellule 17 — Export du modèle final

In [ ]:
# Sauvegarder le modèle final dans Google Drive
from google.colab import drive
drive.mount('/content/drive')

EXPORT_PATH = '/content/drive/MyDrive/fracture_model_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'architecture': 'efficientnet_b3',
    'img_size': IMG_SIZE,
    'val_acc': best_val_acc
}, EXPORT_PATH)

print(f'✅ Modèle exporté : {EXPORT_PATH}')
print(f'   Val Accuracy finale : {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

## Cellule 18 — Résumé final du projet

In [ ]:
print('=' * 60)
print('📋 RÉSUMÉ DU PROJET')
print('=' * 60)
print(f'Architecture    : EfficientNet-B3 (Transfer Learning)')
print(f'Dataset         : Fracture Multi-Region X-ray Data')
print(f'Train images    : {len(train_dataset)}')
print(f'Val images      : {len(val_dataset)}')
print(f'Test images     : {len(test_dataset)}')
print(f'Classes         : {CLASS_NAMES}')
print(f'Meilleure Val Acc : {best_val_acc*100:.2f}%')
print(f'Explainability  : Grad-CAM')
print(f'Rapport         : Généré via LLM (Claude API)')
print('=' * 60)